In [13]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
from tqdm import tqdm
from torchmetrics.image.fid import FrechetInceptionDistance
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [14]:
afhq_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset_path = "../assignment4/data/afhq/train"
train_dataset = datasets.ImageFolder(root=dataset_path, transform=afhq_transforms)
train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True, num_workers=2)
print("AFHQ Dataset initialized successfully for Diffusion.")

AFHQ Dataset initialized successfully for Diffusion.


In [15]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=time.device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn1 = nn.GroupNorm(8, out_ch)
        self.bn2 = nn.GroupNorm(8, out_ch)
        self.relu = nn.SiLU()
        
    def forward(self, x, t):
        h = self.bn1(self.relu(self.conv1(x)))
        time_emb = self.relu(self.time_mlp(t))
        time_emb = time_emb[(..., ) + (None, ) * 2]
        h = h + time_emb
        h = self.bn2(self.relu(self.conv2(h)))
        return h

class DiffusionUNet(nn.Module):
    def __init__(self):
        super().__init__()
        time_dim = 128
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim * 4)
        )
        
        self.inc = nn.Conv2d(3, 64, 3, padding=1)
        self.down1 = Block(64, 128, time_dim * 4)
        self.down2 = Block(128, 256, time_dim * 4)
        self.down3 = Block(256, 512, time_dim * 4)
        
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
        self.up1 = Block(512 + 256, 256, time_dim * 4)
        self.up2 = Block(256 + 128, 128, time_dim * 4)
        self.up3 = Block(128 + 64, 64, time_dim * 4)
        self.outc = nn.Conv2d(64, 3, 3, padding=1)

    def forward(self, x, timestep):
        t = self.time_mlp(timestep)
        x1 = self.inc(x)
        x2 = self.down1(self.pool(x1), t)
        x3 = self.down2(self.pool(x2), t)
        x4 = self.down3(self.pool(x3), t)
        
        # Skip connections
        x = self.up1(torch.cat([self.up(x4), x3], dim=1), t)
        x = self.up2(torch.cat([self.up(x), x2], dim=1), t)
        x = self.up3(torch.cat([self.up(x), x1], dim=1), t)
        return self.outc(x)

In [16]:
timesteps = 1000
beta_start = 1e-4
beta_end = 0.02
betas = torch.linspace(beta_start, beta_end, timesteps).to(device)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, axis=0)

def q_sample(x_start, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x_start)
    sqrt_alphas_cumprod_t = torch.sqrt(alphas_cumprod[t])[:, None, None, None]
    sqrt_one_minus_alphas_cumprod_t = torch.sqrt(1. - alphas_cumprod[t])[:, None, None, None]
    return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise

In [ ]:
model = DiffusionUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
epochs = 50

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for batch_idx, (imgs, _) in enumerate(pbar):
        imgs = imgs.to(device)
        noise = torch.randn_like(imgs)
        
        # Sample random timesteps
        t = torch.randint(0, timesteps, (imgs.shape[0],), device=device).long()
        x_noisy = q_sample(imgs, t, noise=noise)
        
        # Predict the injected noise
        predicted_noise = model(x_noisy, t)
        loss = F.mse_loss(predicted_noise, noise)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})

Epoch 5/50:  91%|███████████████████████████████████████████████████████████████████████▏      | 105/115 [00:18<00:01,  5.62it/s, loss=0.0433]

In [ ]:
@torch.no_grad()
def p_sample_loop(model, shape):
    model.eval()
    b = shape[0]
    img = torch.randn(shape, device=device)
    for i in tqdm(reversed(range(0, timesteps)), desc='Sampling loop', total=timesteps):
        t = torch.full((b,), i, device=device, dtype=torch.long)
        predicted_noise = model(img, t)
        
        alpha = alphas[t][:, None, None, None]
        alpha_cumprod = alphas_cumprod[t][:, None, None, None]
        beta = betas[t][:, None, None, None]
        
        if i > 0:
            noise = torch.randn_like(img)
        else:
            noise = torch.zeros_like(img)
            
        img = 1 / torch.sqrt(alpha) * (img - ((1 - alpha) / (torch.sqrt(1 - alpha_cumprod))) * predicted_noise) + torch.sqrt(beta) * noise
    
    img = (img.clamp(-1, 1) + 1) / 2 # Un-normalize back to [0, 1] for visualization
    return img

# Generate a sample grid
samples = p_sample_loop(model, shape=(16, 3, 64, 64))
grid = make_grid(samples, nrow=4)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.title("Diffusion AFHQ Samples")
plt.axis('off')
plt.show()

# To fully evaluate FID against the VAEs:
# fid = FrechetInceptionDistance(feature=2048).to(device)
# [Feed real normalized batches via fid.update(real_batch, real=True)]
# [Feed generated batches via fid.update(fake_batch, real=False)]
# print(f"Diffusion FID Score: {fid.compute().item()}")